# Assignment 3 — Quantum Pattern Search

## Objective

Build a Quantum Pattern Search application using Qiskit that searches for a selected 3-bit pattern from the eight possible patterns:

`000, 001, 010, 011, 100, 101, 110, 111`

The application uses:

- Superposition
- Oracle-based phase marking
- Interference
- Amplitude amplification
- Grover's Algorithm
- Qiskit's AerSimulator

The oracle is constructed dynamically, so the same application works for any valid 3-bit target.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

SHOTS = 1024
NUM_QUBITS = 3

simulator = AerSimulator()

search_space = [format(i, "03b") for i in range(2 ** NUM_QUBITS)]

print("Search space:")
print(" ".join(search_space))

## 2. Select the Target Pattern

The user can enter any 3-bit pattern.

Examples:

- `000`
- `011`
- `101`
- `110`

The oracle is generated from the selected input; there is no hard-coded target.

In [ ]:
target = input("Enter a target 3-bit pattern (000-111): ").strip()

if len(target) != NUM_QUBITS or any(bit not in "01" for bit in target):
    raise ValueError("Target must be exactly 3 binary digits, for example 101.")

print(f"\nTarget pattern: {target}")
print(f"Search space: {search_space}")

## 3. Dynamic Oracle

The oracle marks the selected target state by applying a phase flip.

For example, if the target is `101`, the oracle changes:

\[
|101\rangle ightarrow -|101\rangle
\]

while leaving the other basis states unchanged.

The implementation below works for **any** 3-bit target entered by the user.

In [ ]:
def apply_oracle(qc, target):
    """Apply a phase oracle that marks the requested 3-bit target."""

    # Qiskit's q0 is the rightmost bit in a measured bitstring.
    # Therefore target[2] corresponds to q0, target[1] to q1,
    # and target[0] to q2.
    target_reversed = target[::-1]

    # Convert the target into |111> so MCX can mark it.
    for qubit, bit in enumerate(target_reversed):
        if bit == "0":
            qc.x(qubit)

    # Multi-controlled X with an H before/after acts as a phase flip.
    qc.h(2)
    qc.mcx([0, 1], 2)
    qc.h(2)

    # Undo the X transformations.
    for qubit, bit in enumerate(target_reversed):
        if bit == "0":
            qc.x(qubit)


# Example: build and display the oracle for the selected target.
oracle_demo = QuantumCircuit(NUM_QUBITS)
apply_oracle(oracle_demo, target)

print(f"Dynamic oracle for target: {target}")
display(oracle_demo.draw("mpl"))

## 4. Grover Diffuser

The diffuser performs the inversion-about-the-mean operation.

It increases the amplitude of the marked state while reducing the amplitudes of the other states.

In [ ]:
def apply_diffuser(qc):
    """Apply the 3-qubit Grover diffusion operator."""

    qc.h(range(NUM_QUBITS))
    qc.x(range(NUM_QUBITS))

    # Phase flip of |111>
    qc.h(2)
    qc.mcx([0, 1], 2)
    qc.h(2)

    qc.x(range(NUM_QUBITS))
    qc.h(range(NUM_QUBITS))

## 5. Build the Complete Grover Circuit

We first place all eight states into an equal superposition using Hadamard gates.

For 8 states and 1 marked state, two Grover iterations provide strong amplitude amplification.

In [ ]:
grover = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)

# Step 1: Create equal superposition
grover.h(range(NUM_QUBITS))

# Step 2: Grover iterations
NUM_ITERATIONS = 2

for _ in range(NUM_ITERATIONS):
    apply_oracle(grover, target)
    apply_diffuser(grover)

print(f"Target pattern: {target}")
print(f"Grover iterations: {NUM_ITERATIONS}")
print("\nQuantum Circuit:")
display(grover.draw("mpl"))

## 6. Run the Circuit Using AerSimulator

The circuit is measured 1024 times.

Because quantum measurement is probabilistic, the target should appear with a high frequency rather than necessarily appearing in every shot.

In [ ]:
# Add measurements without changing the circuit used for displaying the
# unmeasured Grover state.
measurement_circuit = grover.copy()
measurement_circuit.measure_all()

result = simulator.run(
    measurement_circuit,
    shots=SHOTS
).result()

counts = result.get_counts()

print("Measurement counts:")
print(counts)

In [ ]:
print("Measurement Histogram:")

display(
    plot_histogram(
        counts,
        title=f"Grover Search — Target: {target}"
    )
)

## 7. Report the Pattern Identified by the Quantum Algorithm

In [ ]:
identified_pattern = max(counts, key=counts.get)
identified_count = counts[identified_pattern]
success_rate = identified_count / SHOTS * 100

print("========================================")
print("       QUANTUM PATTERN SEARCH")
print("========================================")
print(f"Target pattern:       {target}")
print(f"Identified pattern:   {identified_pattern}")
print(f"Target measurements:  {counts.get(target, 0)} / {SHOTS}")
print(f"Target success rate:  {counts.get(target, 0) / SHOTS * 100:.2f}%")
print("========================================")

if identified_pattern == target:
    print("Result: Target pattern successfully identified.")
else:
    print("Result: The most frequent result was not the target in this run.")

## 8. Verify That the Oracle Is Dynamic

The application should work for different targets without changing the oracle code.

For example, run the notebook again with:

- `000`
- `011`
- `101`
- `110`

Only the user input changes. The same `apply_oracle()` function constructs the appropriate phase-marking operation.

In [ ]:
# Optional verification:
# Uncomment and run this cell to inspect the oracle circuit for every possible target.

for pattern in search_space:
    test_oracle = QuantumCircuit(NUM_QUBITS)
    apply_oracle(test_oracle, pattern)
    print(f"Oracle for {pattern}:")
    display(test_oracle.draw("text"))

## 9. Short Explanation

1. The three qubits are initialized and Hadamard gates create an equal superposition of all eight possible 3-bit patterns.
2. The dynamically generated oracle identifies the selected target by applying a phase flip only to that state.
3. The phase marking itself is not directly visible in ordinary measurement probabilities.
4. The Grover diffuser performs inversion about the mean, causing interference between the amplitudes.
5. Repeating the oracle and diffuser amplifies the target state's amplitude while suppressing the others.
6. After two Grover iterations, the target has a much higher probability of being measured.
7. AerSimulator executes the circuit for 1024 shots and the histogram shows the resulting distribution.
8. The most frequently measured pattern is reported as the pattern identified by the quantum search.

## Conclusion

This application demonstrates the main ideas behind Grover's search algorithm on an eight-element search space. The important feature is that the oracle is **constructed dynamically from the user's target pattern**, allowing the same quantum search application to find different 3-bit patterns without changing the algorithm itself.